# Overview

This project presents two methods to analyze a repository for DORA metrics.

## Method 1: Github Insights export

Method 1 accepts a collection of text files representing a copy-paste of the github insights page. A script is ran that parses the copied insights information into a collection of tables.

## Method 2: Git Log

Method 2 parses the git logfile of a project to extract DORA metrics. https://chanmeng666.medium.com/from-git-log-to-insights-evaluating-team-contributions-in-github-projects-aca645b45ef1

## Method 3: Scrpts + Github API

https://github.com/ministryofjustice/dora-the-explora/tree/main


In [15]:
import pandas as pd
from datetime import datetime, timedelta
import re


def parse_file(file_path: str, date_base:datetime) -> pd.DataFrame:
    """parses a text export of a github insights into pandas dataframe."""

    def _parse_date(date_rel: str, date_base: datetime) -> datetime:
        patterns = [
            "days ago",
            "last week",
            "weeks ago",
            "last month",
            "on"]
        index = -1
        for i, pattern in enumerate(patterns):
            if re.search(pattern, date_rel):
                index = i
                break

        if index == 0:
            num = int(date_rel.split()[0])
            date = date_base - timedelta(days=num)
        elif index == 1:
            date = date_base - timedelta(days=7)
        elif index == 2:
            num = int(date_rel.split()[0])
            date = date_base - timedelta(weeks=num)
        elif index == 3:
            date = date_base - timedelta(weeks=4)
        elif index == 4:
            # I dont want to code this part for the handful of cases
            date = date_base - timedelta(weeks=4)
        else:
            # No pattern matched
            date = date_base
        return date

    file_contents = ""

    with open(file_path) as file:
        file_contents = file.read()
    
    file_arr = file_contents.splitlines()
    file_arr = file_arr[1:]
    
    df = pd.DataFrame(
        [file_arr[i:i+3] for i in range(0, len(file_arr), 3)],
        columns=['msg', 'hash', 'date_rel']
    )
    df['date_rel'] = df['hash'].apply(lambda x: " ".join(x.split()[2:]))
    df['hash'] = df['hash'].apply(lambda x: x.split()[0])
    df['date_abs'] = df['date_rel'].apply(_parse_date, args=(date_base,))


    return df


parse_file("langchain_closed_requests.txt", datetime.today()).head()

,msg,hash,date_rel,date_abs
0,langchain-tests @1.0.2 uses `blockbuster` but ...,#34110,2 days ago,2025-11-27 19:36:57.483549
1,ChatOpenAI封装本地vllm服务链式调用出错,#34067,3 days ago,2025-11-26 19:36:57.483549
2,DOC: <Issue related to /docs/tutorials/chatbot...,#34099,3 days ago,2025-11-26 19:36:57.483549
3,When tools are bound in the `create_agent` fun...,#34082,3 days ago,2025-11-26 19:36:57.483549
4,Add TOON (Token Oriented Object Notation) support,#34059,4 days ago,2025-11-25 19:36:57.483549


In [ ]:
FILES = {"langchain_closed_requests.txt": "closed_requests.csv",
         "langchain_merged_requests.txt": "merged_requests.csv",
         "langchain_opened_issues.txt": "opened_issues.csv",
         "langchain_opened_requests.txt": "opened_requests.csv",
         "langchain_releases.txt": "releases.csv"}
DATE = datetime(2025, 11, 29)

for source, target in FILES.items():
    parse_file(source, DATE).to_csv(target)

AttributeError: 'NoneType' object has no attribute 'split'